# Airline s6e10 – nhóm D (TE có điều kiện, GPU)

1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Runtime → **Run all**. Ô thứ 2 hiện nút **Choose Files** → chọn **`colab_pack2.zip`** (file MỚI, ~30 MB).
3. Chạy xong, ô cuối tự tải **`out_D_ctx.zip`** về máy → báo lại.

Mỗi mô hình ~30–60 phút. Muốn lấy phần đã xong giữa chừng: chạy riêng ô cuối.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import files
if not os.path.exists('/content/colab_pack2.zip'):
    up = files.upload()   # chọn colab_pack2.zip
    name = list(up)[0]
    if name != 'colab_pack2.zip': os.rename(name, 'colab_pack2.zip')
!cd /content && unzip -o -q colab_pack2.zip && ls -la /content/AS /content/AS/src

In [ ]:
!pip install -q catboost xgboost lightgbm

In [ ]:
!cd /content/AS/src && python train.py cat_ctx --threads 2 --seed 0 --tag cat_ctx_gpu --params '{"task_type":"GPU"}' 2>&1 | grep -E 'fold|OOF|Error|Trace|\('

In [ ]:
!cd /content/AS/src && python train.py xgb_ctx --threads 2 --seed 1 --tag xgb_ctx_d10_gpu --params '{"device":"cuda","max_depth":10,"learning_rate":0.015}' 2>&1 | grep -E 'fold|OOF|Error|Trace|\('

In [ ]:
!cd /content/AS/src && python train.py cat_ctx --threads 2 --seed 1 --tag cat_ctx_d10_gpu --params '{"task_type":"GPU","depth":10,"learning_rate":0.04}' 2>&1 | grep -E 'fold|OOF|Error|Trace|\('

## Tải kết quả (chạy lại ô này bất cứ lúc nào)

In [ ]:
!cat /content/AS/artifacts/zoo/scores.jsonl
!cd /content/AS/artifacts && rm -f /content/out_D_ctx.zip && zip -q -r /content/out_D_ctx.zip zoo
from google.colab import files; files.download('/content/out_D_ctx.zip')